# Self-Correcting Agentic Retrieval Loops

Most retrieval agents run the same pipeline on every query. That overspends on easy
questions and underserves hard ones. This notebook builds a **self-correcting loop**
instead: retrieve once with hybrid search, judge the result with a **cheap signal**,
and escalate to a more expensive action only when the signal says retrieval was weak.

The lesson is a **method**, not a fixed recipe. The signals that work here are specific
to this corpus; the way you find them transfers to yours:

1. Define what "good retrieval" means for your task.
2. Engineer cheap candidate signals from the retrieval result.
3. Benchmark which signals predict weak evidence on your data.
4. Turn the winning signal into a gate, and route each query to the cheapest action that fixes it.
5. Measure quality and cost together.

**What "good retrieval" means here.** The agent answers from a **top-3 answer context**:
it reads only the first three retrieved passages. So good retrieval means the supporting
evidence lands in that top-3 window. A cheap signal tells you when it didn't, before you
spend an LLM call finding out.

> This notebook is the runnable companion to the [Self-Correcting Retrieval Loops tutorial](https://qdrant.tech/documentation/tutorials-search-engineering/self-correcting-retrieval-loops/).
> It builds two small collections from a curated MuSiQue slice so the whole loop runs in
> a few minutes. The headline cost and quality numbers near the end are **reported from a
> run on the full workload**, not recomputed on the slice.

## Setup

Install the client, the local embedders, and the agent's LLM client.

In [ ]:
%pip install -q "qdrant-client[fastembed]" litellm scikit-learn pandas numpy matplotlib

This notebook uses a **Qdrant Cloud** cluster to hold the collections and generates all
embeddings **locally with FastEmbed**. The signals read raw, per-retriever score views
(`bge-base` dense cosines, miniCOIL ranks, ColBERT MaxSim), so the exact models matter:
they are what the cited results were measured on. Create a free cluster at
[cloud.qdrant.io](https://cloud.qdrant.io/), then set `QDRANT_URL`, `QDRANT_API_KEY`, and
`ANTHROPIC_API_KEY` as Colab secrets (the key icon in the left sidebar).

In [ ]:
import os, json, re, string, statistics, urllib.request
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import litellm
import matplotlib.pyplot as plt
from qdrant_client import QdrantClient, models
from fastembed import TextEmbedding, SparseTextEmbedding, LateInteractionTextEmbedding
from sklearn.metrics import roc_auc_score

# Credentials. On Colab, set these as secrets (key icon). Get the values from cloud.qdrant.io.
try:
    from google.colab import userdata
    QDRANT_URL = userdata.get("QDRANT_URL")
    QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    QDRANT_URL = os.environ.get("QDRANT_URL", "http://localhost:6333")
    QDRANT_API_KEY = os.environ.get("QDRANT_API_KEY")

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=120)

# Collection schema, model ids, and retrieval sizes (the whole config in one place).
COLLECTION = "musique"                                 # dense (bge) + miniCOIL sparse
COLBERT_COLLECTION = "musique_colbert"                 # dense (reused) + ColBERT multivector
DENSE_MODEL = "BAAI/bge-base-en-v1.5"                  # 768-d dense, cosine
MINICOIL_MODEL = "Qdrant/minicoil-v1"                  # word-sense-aware sparse, IDF modifier
COLBERT_MODEL = "answerdotai/answerai-colbert-small-v1"
DENSE_VEC, MINICOIL_VEC, COLBERT_VEC = "dense", "minicoil", "colbert"
RETRIEVE_N = 50          # per-retriever prefetch depth before fusion
TOP_K = 10               # signal / pool window
ANSWER_K = 3             # focused passages the LLM reads to answer
AGENT_MODEL = "anthropic/claude-sonnet-4-6"            # decompose + answer
FAST_MODEL = "anthropic/claude-haiku-4-5"              # the fast sufficiency autorater (STOP)

# The query-side embedders (local, ONNX). Warm them so later timings are clean.
dense_model = TextEmbedding(DENSE_MODEL)
minicoil_model = SparseTextEmbedding(MINICOIL_MODEL)
colbert_model = LateInteractionTextEmbedding(COLBERT_MODEL)
for warm in (dense_model, minicoil_model):
    next(iter(warm.query_embed("warm up")))
next(iter(colbert_model.query_embed("warm up")))
print("models ready")

### The curated corpus slice

This pulls a few hundred MuSiQue passages and a mixed set of questions (single-hop,
multi-hop, and unanswerable). The questions carry their split label, so the calibration
set used to benchmark signals is available directly.

In [ ]:
# The slice ships with this notebook. Locally these files already sit next to it; on Colab
# they download once.
DATA_REPO = "https://raw.githubusercontent.com/qdrant/examples/master/self-correcting-retrieval-loops"  # TODO: confirm path when the examples PR merges
for fn in ("slice_corpus.jsonl", "slice_questions.jsonl",
           "headline_final_v25.json", "targeted_stop_v25.json"):
    if not Path(fn).exists():
        urllib.request.urlretrieve(f"{DATA_REPO}/{fn}", fn)

corpus = [json.loads(l) for l in open("slice_corpus.jsonl")]
questions = [json.loads(l) for l in open("slice_questions.jsonl")]
by_id = {q["id"]: q for q in questions}

def doc_embed_text(d):
    title, text = (d.get("title") or "").strip(), (d.get("text") or "").strip()
    return f"{title}. {text}" if title else text

print(f"{len(corpus)} passages, {len(questions)} questions "
      f"({sum(q['split']=='calibration' for q in questions)} calibration)")

### Build the two collections

One collection holds the **hybrid baseline** (dense + miniCOIL sparse). A second holds the
**ColBERT** multivector used by one of the corrective actions. Both reuse the same dense
vectors. The named-vector and multivector mechanics are covered in the
[multivector tutorial](https://qdrant.tech/documentation/tutorials-search-engineering/using-multivector-representations/);
here we just build them.

In [ ]:
def to_sparse(emb):
    return models.SparseVector(indices=emb.indices.tolist(), values=emb.values.tolist())

def batched_upsert(coll, points, batch_size=128):
    for i in range(0, len(points), batch_size):
        client.upsert(coll, points[i:i + batch_size], wait=True)

def build_collections():
    texts = [doc_embed_text(d) for d in corpus]
    dense_vecs = [v.tolist() for v in dense_model.embed(texts, batch_size=64)]
    minicoil_vecs = list(minicoil_model.embed(texts, batch_size=64))
    colbert_vecs = [[r.tolist() for r in v] for v in colbert_model.embed(texts, batch_size=32)]

    if client.collection_exists(COLLECTION):
        client.delete_collection(COLLECTION)
    client.create_collection(
        COLLECTION,
        vectors_config={DENSE_VEC: models.VectorParams(size=768, distance=models.Distance.COSINE)},
        sparse_vectors_config={MINICOIL_VEC: models.SparseVectorParams(modifier=models.Modifier.IDF)},
    )
    if client.collection_exists(COLBERT_COLLECTION):
        client.delete_collection(COLBERT_COLLECTION)
    client.create_collection(
        COLBERT_COLLECTION,
        vectors_config={
            DENSE_VEC: models.VectorParams(size=768, distance=models.Distance.COSINE),
            COLBERT_VEC: models.VectorParams(
                size=96, distance=models.Distance.COSINE,
                multivector_config=models.MultiVectorConfig(comparator=models.MultiVectorComparator.MAX_SIM),
                hnsw_config=models.HnswConfigDiff(m=0)),  # rescorer only; reached via the dense prefetch
        },
    )
    base, colb = [], []
    for i, d in enumerate(corpus):
        payload = {"title": d["title"], "text": d["text"]}
        base.append(models.PointStruct(id=d["doc_id"],
            vector={DENSE_VEC: dense_vecs[i], MINICOIL_VEC: to_sparse(minicoil_vecs[i])}, payload=payload))
        colb.append(models.PointStruct(id=d["doc_id"],
            vector={DENSE_VEC: dense_vecs[i], COLBERT_VEC: colbert_vecs[i]}, payload=payload))
    batched_upsert(COLLECTION, base)
    batched_upsert(COLBERT_COLLECTION, colb)

want = len(corpus)
have = client.count(COLLECTION).count if client.collection_exists(COLLECTION) else -1
if have != want:
    build_collections()
print(f"'{COLLECTION}': {client.count(COLLECTION).count} | "
      f"'{COLBERT_COLLECTION}': {client.count(COLBERT_COLLECTION).count}")

### The LLM call

The agent uses an LLM for three jobs: generating a decomposition sub-question, writing the
final answer, and deciding when the evidence is enough. All three go through one helper set
to deterministic output, with retries for transient API errors.

In [ ]:
def ask_llm(system, user, max_tokens=256, model=AGENT_MODEL, temperature=0.0):
    litellm.suppress_debug_info = True
    resp = litellm.completion(
        model=model,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
        max_tokens=max_tokens, temperature=temperature, timeout=45, num_retries=3,
    )
    return (resp.choices[0].message.content or "").strip()

## The hybrid baseline, and where it breaks

The workload has three kinds of questions:

| kind | what has to happen |
|---|---|
| single-hop | one retrieved passage contains the answer |
| multi-hop | the first passage reveals the next thing to look up |
| unanswerable | the corpus lacks the evidence, so the agent should abstain |

The baseline is one **hybrid** retrieval. Qdrant gets candidates two ways (dense vectors for
semantic matches, miniCOIL sparse for lexical matches) and fuses the ranks server-side with
[Reciprocal Rank Fusion (RRF)](https://qdrant.tech/documentation/search/hybrid-queries/). No
cross-encoder in the baseline: the signals read the fusion scores directly.

In [ ]:
def embed(text):
    dense = next(iter(dense_model.query_embed(text))).tolist()
    minicoil = next(iter(minicoil_model.query_embed(text)))
    return dense, minicoil

def hybrid_search(question, limit=TOP_K, enc=None):
    dense, minicoil = enc or embed(question)
    return client.query_points(
        COLLECTION,
        prefetch=[
            models.Prefetch(query=dense, using=DENSE_VEC, limit=RETRIEVE_N),
            models.Prefetch(query=to_sparse(minicoil), using=MINICOIL_VEC, limit=RETRIEVE_N),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=limit, with_payload=True,
    ).points

@dataclass
class Passage:
    doc_id: str
    title: str
    text: str
    score: float

def to_passages(points):
    return [Passage(p.id, p.payload["title"], p.payload["text"], p.score) for p in points]

def show_hits(hits, gold_ids, k=3, snippet=95):
    gold = set(gold_ids)
    for rank, h in enumerate(hits[:k], 1):
        did = h.id if hasattr(h, "id") else h.doc_id
        title = h.payload["title"] if hasattr(h, "payload") else h.title
        text = h.payload["text"] if hasattr(h, "payload") else h.text
        print(f"  [{'GOLD' if did in gold else '    '}] #{rank}  {title}")
        print(f"          {' '.join((text or '').split())[:snippet]}...")

**Single-hop: the cheap path is enough.** One passage carries the answer, and hybrid retrieval
puts it in the top-3 answer context.

In [ ]:
single = by_id["2hop__101521_42157__h0"]
print(f"Q: {single['question']}\n")
show_hits(hybrid_search(single["question"]), single["gold_doc_ids"])

**Multi-hop: the baseline breaks.** The first retrieval can only find the bridge entity. The
passage that answers the question is never retrieved from the question as written,
so no amount of reordering recovers it. This is a recall problem, not a ranking problem.

In [ ]:
multi = by_id["2hop__615262_131886"]
gold = set(multi["gold_doc_ids"])
print(f"Q: {multi['question']}\n")
print("baseline retrieve, question as written (top-3):")
show_hits(hybrid_search(multi["question"]), gold)
print(f"\ngold passages in the top-3: {len({p.id for p in hybrid_search(multi['question'])[:3]} & gold)} of {len(gold)}")

## Signals: predicting weak retrieval cheaply

A **signal** is a low-cost metric computed from the retrieval result that predicts whether the
evidence is weak, before the agent spends anything to find out. A useful signal has to do two
things: separate good retrievals from weak ones on calibration data, and be cheap enough to run
on every query.

The plan: on calibration data we know whether all the gold landed in the top-3. We engineer
candidate signals, score each by how well it separates good from weak (its AUC), keep the ones
that clear a bar, and drop near-duplicates.

### The candidates, by family

We test five candidates. The split that matters is whether a signal reads the already-fused
hybrid result or asks Qdrant for one extra raw retriever view.

| family | signal | what it reads | flags weak when |
|---|---|---|---|
| height | `max_score` | top-1 fused score | low |
| spread (fused) | `score_variance` | spread of the fused top-k scores | low (flat ranking) |
| spread (raw) | `dense_variance` | spread of the raw dense cosines | low (dense can't separate its hits) |
| coverage | `evidence_coverage` | question entities present in the top-k text | low (text misses them) |
| agreement | `retriever_divergence` | dense vs miniCOIL top-k overlap | high (they disagree) |

RRF combines ranks well but flattens score shape, so we test both the fused score spread (free)
and the raw dense cosine spread (one extra Qdrant query, no extra model call).

In [ ]:
def dense_ranking(question, enc=None):
    # the RAW dense cosine ranking, pre-fusion: [(doc_id, cosine), ...]
    dense, _ = enc or embed(question)
    pts = client.query_points(COLLECTION, query=dense, using=DENSE_VEC, limit=TOP_K, with_payload=False).points
    return [(p.id, p.score) for p in pts]

def minicoil_ranking(question, enc=None):
    # the RAW miniCOIL ranking, pre-fusion: [doc_id, ...]
    _, minicoil = enc or embed(question)
    pts = client.query_points(COLLECTION, query=to_sparse(minicoil), using=MINICOIL_VEC,
                              limit=TOP_K, with_payload=False).points
    return [p.id for p in pts]

_QUESTION_STOP = {"what", "who", "whom", "whose", "where", "when", "which", "why", "how",
                  "is", "was", "are", "were", "did", "do", "does", "the", "a", "an", "name",
                  "in", "of", "on", "at", "to", "for", "by", "as", "that", "this"}
_NAME_CONNECTORS = {"of", "the", "and", "de", "von", "van", "del", "la", "el", "da", "di", "&"}

def question_entities(question):
    # the capitalized spans + 4-digit years in the question, e.g. {"pulp fiction", "1994"}
    toks = (question or "").split()
    ents, cur = [], []
    for i, tok in enumerate(toks):
        w = tok.strip(string.punctuation)
        if not w:
            if cur: ents.append(" ".join(cur)); cur = []
            continue
        if w[0].isupper() and not (i == 0 and w.lower() in _QUESTION_STOP):
            cur.append(w)
        elif cur and w.lower() in _NAME_CONNECTORS and i + 1 < len(toks) \
                and toks[i + 1].strip(string.punctuation)[:1].isupper():
            cur.append(w)
        elif cur:
            ents.append(" ".join(cur)); cur = []
    if cur: ents.append(" ".join(cur))
    out = {e.lower() for e in ents if len(e) >= 2}
    out |= set(re.findall(r"\b\d{4}\b", question or ""))
    return out

Now the five candidate signals. `retrieve_signals` runs the shared reads once (one encode, the
fused hybrid result, and the two raw single-retriever rankings); each signal then reads one
value from those results.

In [ ]:
def retrieve_signals(question, enc=None):
    enc = enc or embed(question)
    return hybrid_search(question, enc=enc), dense_ranking(question, enc=enc), minicoil_ranking(question, enc=enc)

def max_score(fused):       return fused[0].score
def score_variance(fused):  return statistics.pstdev([p.score for p in fused])
def dense_variance(dense):  return statistics.pstdev([s for _, s in dense])

def evidence_coverage(question, fused):
    ents = question_entities(question)
    if not ents:
        return 1.0
    blob = " ".join(f"{p.payload['title']} {p.payload['text']}" for p in fused).lower()
    return sum(1 for e in ents if e in blob) / len(ents)

def retriever_divergence(dense, sparse_ids):
    dense_ids = [i for i, _ in dense]
    return 1.0 - len(set(dense_ids) & set(sparse_ids)) / max(len(dense_ids), len(sparse_ids))

### Benchmark the candidates, live

We score each signal on the calibration questions. The label is `full_gold@3`: did **all** the
supporting passages land in the top-3 answer context? AUC reads as a separation score: 0.5 is
chance, 1.0 is perfect. This runs against Qdrant only, no LLM calls.

In [ ]:
cal_questions = [q for q in questions
                 if q["split"] == "calibration" and q.get("answerable") and q.get("gold_doc_ids")]

def feature_row(q):
    fused, dense, sparse_ids = retrieve_signals(q["question"])
    gold = set(q["gold_doc_ids"])
    return {
        "full_gold_label": 1 if gold.issubset({p.id for p in fused[:ANSWER_K]}) else 0,
        "dense_variance": dense_variance(dense), "score_variance": score_variance(fused),
        "max_score": max_score(fused), "evidence_coverage": evidence_coverage(q["question"], fused),
        "retriever_divergence": retriever_divergence(dense, sparse_ids),
    }

calibration = [feature_row(q) for q in cal_questions]
labels = [r["full_gold_label"] for r in calibration]
print(f"benchmarked {len(calibration)} calibration questions: "
      f"{sum(labels)} good / {len(labels) - sum(labels)} weak retrievals")

Keep signals with separation at least **0.65**, then drop any that is a near-duplicate
(absolute correlation above **0.85**) of a stronger kept signal.

In [ ]:
AUC_BAR, CORR_MAX = 0.65, 0.85
SIGNALS = ["dense_variance", "score_variance", "max_score", "evidence_coverage", "retriever_divergence"]

def signal_auc(name):
    raw = roc_auc_score(labels, [r[name] for r in calibration])
    return max(raw, 1 - raw)        # separation strength, regardless of direction

def abs_corr(a, b):
    return abs(float(np.corrcoef([r[a] for r in calibration], [r[b] for r in calibration])[0, 1]))

aucs = {name: signal_auc(name) for name in SIGNALS}
kept = []
for name in sorted((n for n in aucs if aucs[n] >= AUC_BAR), key=lambda n: -aucs[n]):
    if all(abs_corr(name, k) <= CORR_MAX for k in kept):
        kept.append(name)
kept = set(kept)

pd.DataFrame([{"signal": n, "AUC": round(aucs[n], 3), "verdict": "kept" if n in kept else "dropped"}
              for n in sorted(aucs, key=lambda n: -aucs[n])])

The same benchmark as a picture: when the good and weak boxes pull apart, the signal separates;
when they overlap, it doesn't.

In [ ]:
def plot_signal_separation(features, aucs, kept):
    good = lambda col: [r[col] for r in features if r["full_gold_label"] == 1]
    weak = lambda col: [r[col] for r in features if r["full_gold_label"] == 0]
    order = sorted(aucs, key=lambda s: -aucs[s])
    fig, axes = plt.subplots(1, len(order), figsize=(14, 3.4))
    for ax, name in zip(axes, order):
        b = ax.boxplot([good(name), weak(name)], tick_labels=["good", "weak"],
                       widths=0.6, patch_artist=True, showfliers=False)
        b["boxes"][0].set(facecolor="#1f9d55", alpha=0.55)
        b["boxes"][1].set(facecolor="#d64545", alpha=0.55)
        ax.set_title(f"{name}\nAUC {aucs[name]:.2f} ({'kept' if name in kept else 'dropped'})", fontsize=9)
        ax.tick_params(labelsize=8)
    fig.suptitle("Each signal on good vs weak retrievals: separation = predictive power", fontsize=11)
    fig.tight_layout(rect=[0, 0, 1, 0.92]); plt.show()

plot_signal_separation(calibration, aucs, kept)

Two signals survive: `dense_variance` (spread of the raw dense cosines) and `score_variance`
(spread of the fused RRF scores). Low spread means the retriever couldn't separate its top hits,
which is what weak retrieval looks like. The other three either don't separate well enough or
duplicate a stronger signal.

**This is the part that transfers, not the winners.** On a different corpus the coverage or
divergence signal might win instead. Engineer candidates, benchmark them against labeled
retrieval outcomes on your data, and keep what separates. In production, pick signals on one
split, tune the threshold on another, and keep a test split held out.

### The gate: one signal, one floor

The gate turns a signal into a decision: answer from the current result, or escalate. Each
signal gets a floor; below it, retrieval is treated as weak. Raising the floor catches more weak
retrievals but escalates more queries. `youden_floor` picks the threshold that best balances
catching weak retrievals against false alarms.

In [ ]:
weak = [r["full_gold_label"] == 0 for r in calibration]   # True = retrieval was weak

def youden_floor(values):
    best = None
    for thr in sorted(set(values)):
        pred = [v < thr for v in values]
        tp = sum(p and w for p, w in zip(pred, weak)); fp = sum(p and not w for p, w in zip(pred, weak))
        fn = sum((not p) and w for p, w in zip(pred, weak)); tn = sum((not p) and not w for p, w in zip(pred, weak))
        tpr = tp / (tp + fn) if (tp + fn) else 0.0
        fpr = fp / (fp + tn) if (fp + tn) else 0.0
        if best is None or (tpr - fpr) > best[1]:
            best = (thr, tpr - fpr)
    return best[0]

DV_FLOOR = youden_floor([r["dense_variance"] for r in calibration])
SV_FLOOR = youden_floor([r["score_variance"] for r in calibration])
print(f"calibrated floors:  dense_variance < {DV_FLOOR:.5f}   score_variance < {SV_FLOOR:.5f}\n")

fires = [r["dense_variance"] < DV_FLOOR for r in calibration]
tp = sum(f and w for f, w in zip(fires, weak)); fp = sum(f and not w for f, w in zip(fires, weak))
fn = sum((not f) and w for f, w in zip(fires, weak))
print("at the dense_variance floor:")
print(f"  precision       = {tp / (tp + fp):.3f}   (of the queries we escalate, how many were truly weak)")
print(f"  recall          = {tp / (tp + fn):.3f}   (of the truly weak queries, how many we catch)")
print(f"  escalation rate = {sum(fires) / len(fires):.2f}   (fraction of queries sent past the cheap path)")

The histogram shows where the floor sits and what it buys: how much of the weak (red) mass falls
below the line (caught) versus how much of all traffic falls below it (escalated).

In [ ]:
def plot_gate(features, floor, signal="dense_variance", desc="spread of the raw dense scores"):
    good = [r[signal] for r in features if r["full_gold_label"] == 1]
    weakv = [r[signal] for r in features if r["full_gold_label"] == 0]
    vals = np.array([r[signal] for r in features])
    is_weak = np.array([r["full_gold_label"] == 0 for r in features])
    below = vals < floor
    recall = (below & is_weak).sum() / max(is_weak.sum(), 1)
    esc = below.mean()
    fig, ax = plt.subplots(figsize=(8.5, 4.2))
    bins = np.linspace(0, vals.max(), 26)
    ax.hist(good, bins=bins, alpha=0.6, color="#1f9d55", label="good (full gold present)")
    ax.hist(weakv, bins=bins, alpha=0.6, color="#d64545", label="weak (full gold missing)")
    ax.axvspan(0, floor, color="black", alpha=0.06)
    ax.axvline(floor, color="black", ls="--", lw=1.6, label=f"gate floor = {floor:.3f}")
    ax.set_title(f"Low spread predicts weak retrieval\n{signal} floor {floor:.3f}  ->  "
                 f"catches {recall:.0%} of weak retrievals, escalates {esc:.0%} of all queries", fontsize=10)
    ax.set_xlabel(f"{signal}  ({desc})"); ax.set_ylabel("queries")
    ax.legend(fontsize=8); fig.tight_layout(); plt.show()

plot_gate(calibration, DV_FLOOR)

Wire the gate the loop will call. It reads `score_variance` from the fused result and
`dense_variance` from the dense-only ranking; if either falls below its floor, the retrieval is
weak.

In [ ]:
def retrieval_is_weak(fused, dense):
    return score_variance(fused) < SV_FLOOR or dense_variance(dense) < DV_FLOOR

> **Targeting recall instead of balance.** The Youden floor balances catches against false alarms.
> If a missed weak retrieval costs more than an extra escalation, raise the floor to hit a recall
> target (say, catch 90% of weak retrievals) at the price of escalating more. The loop below keeps
> the balanced floors.

## The corrective loop

The gate says *when* retrieval is weak. The loop decides *what to do* about it, matching the fix
to the failure mode:

- **Weak single-hop lookup:** the right passage exists but ranks too low. Fix the ranking.
- **Weak multi-hop query:** the next passage isn't reachable from the question as written. Recover
  the missing evidence.

### Action: ColBERT late interaction (single-hop precision)

ColBERT scores query tokens against passage tokens, so it can promote a specific passage that the
pooled dense vector blurred away. We prefetch a dense candidate pool, then rescore with the ColBERT
multivector (MaxSim). The mechanics of late interaction and multivectors are covered in the
[ColBERT how-to](https://qdrant.tech/documentation/fastembed/fastembed-colbert/) and the
[multivector tutorial](https://qdrant.tech/documentation/tutorials-search-engineering/using-multivector-representations/);
here it is just the corrective action the gate routes to.

In [ ]:
def colbert_rerank(question, limit=TOP_K, enc=None):
    dense, _ = enc or embed(question)
    colbert_vecs = [v.tolist() for v in next(iter(colbert_model.query_embed(question)))]
    return client.query_points(
        COLBERT_COLLECTION,
        prefetch=[models.Prefetch(query=dense, using=DENSE_VEC, limit=RETRIEVE_N)],
        query=colbert_vecs, using=COLBERT_VEC, limit=limit, with_payload=True,
    ).points

tier2 = by_id["2hop__82744_23140__h0"]
gold = tier2["gold_doc_ids"]
print(f"Q: {tier2['question']}\n")
print("hybrid retrieve, the answer passage buried:")
show_hits(hybrid_search(tier2["question"]), gold)
print("\nColBERT late interaction, the answer passage promoted:")
show_hits(colbert_rerank(tier2["question"]), gold)

Hybrid retrieval matched the entity in the question and buried the passage with the actual answer.
ColBERT promoted it to the top. Across the full validation set ColBERT adds little on average, but
it earns its keep on the weak single-hop retrievals the gate routes to it. Cross-encoder rerankers
are a viable alternative; ColBERT produced the best results on this corpus.

### Action: decompose (IRCoT) for multi-hop recall

A reranker can only reorder passages already retrieved. When the needed evidence is missing
entirely, the agent decomposes: read the evidence so far, ask the next still-missing sub-question,
retrieve again, and repeat until it has enough. This retrieve-read-ask loop is **IRCoT**.
Decomposition as a standalone technique is covered in
[Hybrid and Multi-Stage Queries](https://qdrant.tech/documentation/search/hybrid-queries/#query-decomposition-for-multi-hop-questions).

In [ ]:
IRCOT_SYSTEM = (
    "You are running an iterative retrieve-and-reason loop to answer a multi-hop "
    "question. Given the main question, the evidence retrieved so far, and the "
    "sub-questions already asked, output the NEXT single sub-question whose answer is "
    "still MISSING and is needed to answer the main question. Make it self-contained: "
    "name entities explicitly, resolving any bridge entity from the evidence so far. "
    "If the evidence already contains everything needed to answer the main question, "
    "reply with exactly: ENOUGH. Output ONLY the sub-question text or ENOUGH - no prose."
)

def evidence_digest(pools, max_docs=6, max_chars=160):
    seen, lines = set(), []
    for pool in pools:
        for c in pool:
            if c.doc_id in seen:
                continue
            seen.add(c.doc_id)
            lines.append(f"- {c.title}: {(c.text or '')[:max_chars]}")
            if len(lines) >= max_docs:
                return "\n".join(lines)
    return "\n".join(lines) if lines else "(none)"

def next_subquery(question, pools, sub_queries):
    user = (
        f"Main question: {question}\n\n"
        f"Evidence so far:\n{evidence_digest(pools)}\n\n"
        "Sub-questions already asked:\n" + ("\n".join(f"- {s}" for s in sub_queries) or "(none)") +
        "\n\nNext sub-question (or ENOUGH):"
    )
    t = (ask_llm(IRCOT_SYSTEM, user, max_tokens=80) or "").strip()
    if not t or t.upper().startswith("ENOUGH"):
        return None
    return re.sub(r"^[\-\d\.\)\s]+", "", t.splitlines()[0]).strip() or None

def union_pool(pools, k=TOP_K):
    # merge the per-hop pools, keep the max score per doc; this is what lets a later hop's
    # passage surface into the final answer context.
    best = {}
    for pool in pools:
        for c in pool:
            if c.doc_id not in best or c.score > best[c.doc_id].score:
                best[c.doc_id] = c
    return sorted(best.values(), key=lambda c: c.score, reverse=True)[:k]

def decompose(question, max_hops=4, hop0=None):
    pools = [hop0 if hop0 is not None else to_passages(hybrid_search(question))]
    sub_queries = []
    for _ in range(max_hops - 1):
        next_q = next_subquery(question, pools, sub_queries)
        if next_q is None:
            break
        sub_queries.append(next_q)
        pools.append(to_passages(hybrid_search(next_q)))
    return union_pool(pools, TOP_K), sub_queries

multi = by_id["2hop__615262_131886"]
gold = multi["gold_doc_ids"]
print(f"Q: {multi['question']}\n")
print("hybrid retrieve (single pass), only the first hop is reachable:")
show_hits(hybrid_search(multi["question"]), gold)
pool, sub_queries = decompose(multi["question"])
print("\ndecompose reads hop 1, then asks the still-missing sub-question:")
for sq in sub_queries:
    print(f"  -> {sq}")
print("\nunioned evidence, the second supporting passage now in context:")
show_hits(pool, gold, k=4)

### Assemble the loop: `solve()`

`solve()` retrieves once, reads the gate, and escalates only as far as needed. The gate decides
*whether* to escalate; a cheap question-shape check (`looks_multi_hop`) decides *how*. The answer
step reads only the top-3 passages and returns `INSUFFICIENT_CONTEXT` when the evidence is thin.

In [ ]:
ANSWER_SYSTEM = (
    "You answer a question using ONLY the numbered context passages provided. "
    "Reply with ONLY the final answer on a single line: a name, date, number, or short "
    "noun phrase, usually one to six words. Do NOT show reasoning or steps, do NOT "
    "restate the question. If the passages do not contain the information needed to "
    "answer, reply with exactly: INSUFFICIENT_CONTEXT"
)

def generate_answer(question, passages):
    if not passages:
        return "INSUFFICIENT_CONTEXT"
    ctx = "\n".join(f"[{i}] {c.title}. {(c.text or '')[:700]}" for i, c in enumerate(passages[:ANSWER_K], 1))
    t = (ask_llm(ANSWER_SYSTEM, f"Context:\n{ctx}\n\nQuestion: {question}\nAnswer (answer only):",
                 max_tokens=150) or "").strip()
    if "INSUFFICIENT_CONTEXT" in t.upper() or not t:
        return "INSUFFICIENT_CONTEXT"
    last = [ln.strip() for ln in t.splitlines() if ln.strip()][-1]
    return re.sub(r"^(answer|the answer is|final answer)[:\-\s]+", "", last, flags=re.I).strip()

def looks_multi_hop(question):
    # cheap, gold-free router: >= 2 named entities or a long question -> likely a missing hop
    return len(question_entities(question)) >= 2 or len(question.split()) >= 12

def solve(question):
    enc = embed(question)
    fused = hybrid_search(question, enc=enc)
    dense = dense_ranking(question, enc=enc)
    if not retrieval_is_weak(fused, dense):
        return to_passages(fused), "cheap path: confident, answer from the hybrid top-3"
    if looks_multi_hop(question):
        pool, sub_queries = decompose(question, hop0=to_passages(fused))
        return pool, f"decompose: weak + multi-hop ({len(sub_queries)} sub-question(s))"
    return to_passages(colbert_rerank(question, enc=enc)), "ColBERT: weak single-hop, rerank"

Three queries, three paths from one agent.

In [ ]:
routing_demos = [
    ("confident single-hop", "2hop__101521_42157__h0"),
    ("weak single-hop",      "2hop__130545_45439__h0"),
    ("multi-hop",            "2hop__615262_131886"),
]
for label, qid in routing_demos:
    q = by_id[qid]
    pool, route = solve(q["question"])
    answer = generate_answer(q["question"], pool)
    found = len({p.doc_id for p in pool[:3]} & set(q["gold_doc_ids"]))
    print(f"[{label}]  {q['question']}")
    print(f"  route:  {route}")
    print(f"  answer: {answer[:72]}")
    print(f"  gold in answer context: {found}/{len(q['gold_doc_ids'])}\n")

### STOP: deciding whether to answer at all

Routing asks "should I spend more retrieval?" STOP asks "should I answer at all?" The default
**gentle stop** is built into `generate_answer`: it answers from the context or returns
`INSUFFICIENT_CONTEXT`, with no extra model call. The stricter option is a separate **sufficiency
judge** (a fast, cheap model) that reads the same top-3 context and decides whether every needed
fact is present. It catches more unanswerables but can over-refuse answerable questions.

In [ ]:
SUFFICIENCY_SYSTEM = (
    "Judge whether the provided context contains enough information to answer the question. "
    "Use ONLY the context. Reply with exactly one word: SUFFICIENT or INSUFFICIENT."
)

def sufficiency_judge(question, passages):
    if not passages:
        return False
    context = "\n".join(f"[{i}] {c.title}. {(c.text or '')[:600]}" for i, c in enumerate(passages, 1))
    user = (f"Question:\n{question}\n\nContext:\n{context}\n\n"
            "Can the question be answered completely from this context? "
            "Reply with exactly one word: SUFFICIENT or INSUFFICIENT.")
    try:
        t = ask_llm(SUFFICIENCY_SYSTEM, user, max_tokens=20, model=FAST_MODEL)
    except Exception:
        return True   # on API failure, keep the gentle-stop behavior
    words = re.findall(r"[A-Z]+", (t or "").upper())
    return "INSUFFICIENT" not in words

One unanswerable question, traced end to end: the first retrieval, the gate, the decompose
sub-questions, the final answer context, and both STOP decisions.

In [ ]:
unanswerable = by_id["2hop__108098_170204"]
question = unanswerable["question"]
print(f"Q: {question}\n")

enc = embed(question)
fused = hybrid_search(question, enc=enc)
dense = dense_ranking(question, enc=enc)
is_weak = retrieval_is_weak(fused, dense)

print("baseline hybrid retrieve:")
show_hits(fused, [], k=ANSWER_K)
print(f"\ngate: dense_variance={dense_variance(dense):.5f} (floor {DV_FLOOR:.5f}), "
      f"score_variance={score_variance(fused):.5f} (floor {SV_FLOOR:.5f})")
print(f"decision: {'weak -> escalate' if is_weak else 'strong -> answer now'}")

pool, route = solve(question)
print(f"route: {route}")
gentle = generate_answer(question, pool)
sufficient = sufficiency_judge(question, pool[:ANSWER_K])
print(f"\ngentle stop       = {'abstain' if gentle == 'INSUFFICIENT_CONTEXT' else 'answer'} ({gentle})")
print(f"sufficiency judge = {'answer' if sufficient else 'abstain'}")

## Results on the full workload

The numbers below are **reported from a run on the full workload** (321 held-out questions:
single-hop, multi-hop, and unanswerable), not recomputed on the slice above. The slice is a
runnable demo; these are what the method buys at scale.

The answer context is top-3, so the retrieval metrics focus on that small window:

| metric | what it answers |
|---|---|
| recall@3 | did at least one supporting passage reach the answer context? |
| full_gold@3 | did every required passage reach the answer context? |
| MRR | how high did the first supporting passage rank? |
| LLM calls/query | how many LLM calls did the route spend? |
| latency | routing time before the final answer, end to end |

In [ ]:
headline = json.loads(Path("headline_final_v25.json").read_text())["overall"]
rows = []
for name in ("always_answer", "always_colbert", "always_decompose", "ladder"):
    m = headline[name]
    rows.append({
        "policy": name.replace("_", "-"),
        "recall@3": m["recall@3"], "full_gold@3": m["full_gold@3"], "MRR": m["mrr_first"],
        "LLM calls/query": m["llm_calls"], "routing latency (s)": m["avg_latency_s"],
    })
pd.set_option("display.precision", 3)
pd.DataFrame(rows)

The adaptive loop (`ladder`) keeps most of the quality of always-decompose at a fraction of the
LLM calls and latency: it only pays for decomposition on the queries the gate flags. Always-answer
is cheapest but leaves multi-hop quality on the table; always-decompose is the quality ceiling but
spends an LLM call on every query, including the ones that didn't need it.

### STOP, side by side

"Full workload handled" means the agent either answers correctly or correctly refuses an
unanswerable question.

In [ ]:
stop = json.loads(Path("targeted_stop_v25.json").read_text())["variants"]
rows = [
    ("hybrid baseline + gentle",       "baseline_hybrid_gentle"),
    ("loop + gentle (default)",        "ladder_gentle"),
    ("loop + LLM sufficiency check",   "ladder_autorater_all"),
]
pd.DataFrame([{
    "setup": label,
    "catches unanswerable": stop[key]["abstain_unans"],
    "over-refuses answerable": stop[key]["false_stop_ans"],
    "full workload handled": stop[key]["selective_accuracy"],
} for label, key in rows])

### Adapt this to your corpus

1. **Set the answer context.** Decide how many passages the LLM reads. That defines "good retrieval."
2. **Label retrieval outcomes** on a calibration split: did the needed evidence reach that window?
3. **Engineer and benchmark cheap signals.** Keep the ones that separate weak from healthy retrievals on your data.
4. **Match a fix to each failure mode.** Add the cheapest corrective action that addresses it.
5. **Measure quality and cost together.** Keep the loop only where the quality justifies the spend.
6. **Choose STOP deliberately.** Gentle by default; stricter when wrong answers are expensive.

A single signal plus one floor was enough here because the kept signals were redundant. When your
signals carry independent information, fuse them with a small classifier instead of one threshold.
The method is the same: define good retrieval, find signals that predict it on your data, and route
to the cheapest sufficient fix.